## NeRF Reconstruction

In [1]:
import torch
import numpy as np
import open3d as o3d
from pathlib import Path
import sys

# Add parent directory to path for imports
sys.path.append('..')

from utils.nerf_trainer import train_nerf, extract_point_cloud
from utils.colmap_loader import COLMAPDataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
PyTorch version: 2.9.1+cu128
CUDA available: True
CUDA device: NVIDIA RTX 1000 Ada Generation Laptop GPU


## 1. Configure Paths

In [2]:
# Set your paths
colmap_sparse_dir = Path(r"../reconstructions/COLMAP/sparse/0")  # Directory containing cameras.bin, images.bin, points3D.bin
image_dir = Path(r"../images")  # Directory containing the actual images
output_dir = Path(r"../reconstructions/nerf_output")

output_dir.mkdir(parents=True, exist_ok=True)

print(f"COLMAP directory: {colmap_sparse_dir}")
print(f"Image directory: {image_dir}")
print(f"Output directory: {output_dir}")

COLMAP directory: ..\reconstructions\COLMAP\sparse\0
Image directory: ..\images
Output directory: ..\reconstructions\nerf_output


## 2. Load and Inspect COLMAP Data

In [3]:
# Load COLMAP dataset
dataset = COLMAPDataset(str(colmap_sparse_dir), str(image_dir))

print(f"\nDataset Info:")
print(f"Number of cameras: {len(dataset.cameras)}")
print(f"Number of images: {len(dataset.images)}")
print(f"Number of 3D points: {len(dataset.points3D)}")
print(f"\nScene bounds:")
print(f"Min: {dataset.bounds_min}")
print(f"Max: {dataset.bounds_max}")
print(f"Center: {dataset.bounds_center}")
print(f"Radius: {dataset.bounds_radius:.3f}")

Loading COLMAP data...
Loaded 1 cameras
Loaded 20 images
Loaded 7374 3D points
Scene bounds: min=[ -0.18583033 -12.02612023  -0.77016418], max=[ 9.49710068  6.76994614 15.64184649]
Scene center: [ 4.65563518 -2.62808704  7.43584115], radius: 13.382874862232585

Dataset Info:
Number of cameras: 1
Number of images: 20
Number of 3D points: 7374

Scene bounds:
Min: [ -0.18583033 -12.02612023  -0.77016418]
Max: [ 9.49710068  6.76994614 15.64184649]
Center: [ 4.65563518 -2.62808704  7.43584115]
Radius: 13.383


In [4]:
# Visualize COLMAP sparse point cloud
if len(dataset.points3D) > 0:
    points = np.array([p["xyz"] for p in dataset.points3D.values()])
    colors = np.array([p["rgb"] / 255.0 for p in dataset.points3D.values()])
    
    pcd_colmap = o3d.geometry.PointCloud()
    pcd_colmap.points = o3d.utility.Vector3dVector(points)
    pcd_colmap.colors = o3d.utility.Vector3dVector(colors)
    
    print(f"\nCOLMAP sparse point cloud: {len(points)} points")
    o3d.visualization.draw_geometries([pcd_colmap], window_name="COLMAP Sparse Points")
else:
    print("No 3D points in COLMAP reconstruction")


COLMAP sparse point cloud: 7374 points


## 3. Train NeRF Model

This will train both coarse and fine networks using hierarchical sampling.

In [4]:
# Training hyperparameters - Optimized for GPU memory usage
config = {
    'num_epochs': 50,
    'batch_size': 8192,        # Increased batch size to use more GPU memory
    'num_coarse_samples': 64,  # Increase samples for better GPU utilization
    'num_fine_samples': 128,   # Increase samples for better GPU utilization
    'lr': 5e-4,
    'device': 'cuda'
}

print("Training configuration (OPTIMIZED FOR GPU MEMORY USAGE):")
for key, value in config.items():
    print(f"  {key}: {value}")

print("\n🚀 GPU Memory Optimizations:")
print("- Increased batch size to 8192 rays (4x increase)")
print("- Increased sample counts for better GPU utilization")
print("- Reduced CPU data caching")
print("- Optimized tensor operations for GPU")
print("- Gradient accumulation for large effective batch sizes")

Training configuration (OPTIMIZED FOR GPU MEMORY USAGE):
  num_epochs: 50
  batch_size: 8192
  num_coarse_samples: 64
  num_fine_samples: 128
  lr: 0.0005
  device: cuda

🚀 GPU Memory Optimizations:
- Increased batch size to 8192 rays (4x increase)
- Increased sample counts for better GPU utilization
- Reduced CPU data caching
- Optimized tensor operations for GPU
- Gradient accumulation for large effective batch sizes


In [5]:
# Train NeRF
model_coarse, model_fine, dataset = train_nerf(
    colmap_dir=str(colmap_sparse_dir),
    image_dir=str(image_dir),
    output_dir=str(output_dir),
    num_epochs=config['num_epochs'],
    batch_size=config['batch_size'],
    num_coarse_samples=config['num_coarse_samples'],
    num_fine_samples=config['num_fine_samples'],
    lr=config['lr'],
    device=config['device']
)

Loading COLMAP dataset...
Loading COLMAP data...
Loaded 1 cameras
Loaded 20 images
Loaded 7374 3D points
Scene bounds: min=[ -0.18583033 -12.02612023  -0.77016418], max=[ 9.49710068  6.76994614 15.64184649]
Scene center: [ 4.65563518 -2.62808704  7.43584115], radius: 13.382874862232585
Train images: 17, Test images: 3
Creating ray dataset...
Processing 17 images for ray dataset...
Ray dataset created: 8160000 rays
Initializing models...
Ray dataset created: 8160000 rays
Initializing models...


c:\Users\gnoceras\Documents\GustavoPersonal\ReconstructionStudies\notebooks\..\utils\nerf_trainer.py:214: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if device == 'cuda' else None


Sampling bounds: near=6.691, far=33.457
Starting training...


Epoch 1/50:   0%|          | 0/997 [00:00<?, ?it/s]c:\Users\gnoceras\Documents\GustavoPersonal\ReconstructionStudies\notebooks\..\utils\nerf_trainer.py:241: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(scaler is not None)):
c:\Users\gnoceras\Documents\GustavoPersonal\ReconstructionStudies\notebooks\..\utils\nerf_trainer.py:241: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(scaler is not None)):
Epoch 1/50:   0%|          | 3/997 [01:14<6:49:51, 24.74s/it, loss=0.134, loss_fine=0.0675]



KeyboardInterrupt: 

## 4. Extract Point Cloud

Sample the trained NeRF on a 3D grid and extract points with high density.

In [ ]:
# Point cloud extraction parameters
extraction_config = {
    'resolution': 128,           # Grid resolution (higher = more points, slower)
    'density_threshold': 10.0,   # Minimum density for a point to be included
}

print("Extraction configuration:")
for key, value in extraction_config.items():
    print(f"  {key}: {value}")

In [ ]:
# Extract point cloud from fine model
output_ply = output_dir / "nerf_point_cloud.ply"

pcd_nerf = extract_point_cloud(
    model=model_fine,
    dataset=dataset,
    output_path=str(output_ply),
    resolution=extraction_config['resolution'],
    density_threshold=extraction_config['density_threshold'],
    device=config['device']
)

print(f"\nPoint cloud saved to: {output_ply}")

## 5. Visualize Results

In [ ]:
# Visualize NeRF point cloud
o3d.visualization.draw_geometries([pcd_nerf], window_name="NeRF Point Cloud")

In [ ]:
# Compare with COLMAP sparse points
if len(dataset.points3D) > 0:
    # Color COLMAP points differently
    pcd_colmap_copy = o3d.geometry.PointCloud(pcd_colmap)
    pcd_colmap_copy.paint_uniform_color([1, 0, 0])  # Red for COLMAP
    
    # Downsample NeRF for visualization if too dense
    pcd_nerf_vis = pcd_nerf.voxel_down_sample(voxel_size=0.01)
    
    print(f"\nVisualization:")
    print(f"  Red: COLMAP sparse points ({len(pcd_colmap.points)} points)")
    print(f"  Colored: NeRF reconstruction ({len(pcd_nerf_vis.points)} points, downsampled)")
    
    o3d.visualization.draw_geometries(
        [pcd_colmap_copy, pcd_nerf_vis],
        window_name="COLMAP (red) vs NeRF (colored)"
    )

## 6. Point Cloud Statistics

In [ ]:
print("\n=== Point Cloud Statistics ===")
print(f"\nNeRF Point Cloud:")
print(f"  Number of points: {len(pcd_nerf.points)}")
print(f"  Has colors: {pcd_nerf.has_colors()}")
print(f"  Bounds: {pcd_nerf.get_min_bound()} to {pcd_nerf.get_max_bound()}")

if len(dataset.points3D) > 0:
    print(f"\nCOLMAP Sparse Points:")
    print(f"  Number of points: {len(pcd_colmap.points)}")
    print(f"  Density increase: {len(pcd_nerf.points) / len(pcd_colmap.points):.1f}x")

## 7. Optional: Load Saved Model

In [ ]:
# To load a previously trained model:
from utils.nerf_model import NeRFModel

checkpoint_path = output_dir / "nerf_final.pth"

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path)
    
    model_coarse_loaded = NeRFModel().to(config['device'])
    model_fine_loaded = NeRFModel().to(config['device'])
    
    model_coarse_loaded.load_state_dict(checkpoint['model_coarse_state_dict'])
    model_fine_loaded.load_state_dict(checkpoint['model_fine_state_dict'])
    
    print(f"Loaded model from {checkpoint_path}")
else:
    print(f"No checkpoint found at {checkpoint_path}")